# Thesis: Reclaimed Timber in Deep Generative Design
**Notebook ID:** 24_25_optimizer_workflow_with_cost_and_ILP  
**Author:** Jasper Cluistra   
**Last Updated:** 2026-02-27
### Properties script
**Goal:** to generate a cost matrix for the geometry with use of the timber datasets, then using ILP to find the best matches   
**Inputs:**
*   CSV timber dataset
*   Digital geometry

**Outputs:**
*   Best match for each element in a structure

# IMPORTING

## Dataset

In [17]:
from config import RAW_DATA_PATH
import pandas as pd

# Laad de geëxporteerde dataset in je nieuwe omgeving
# Voeg sep=';' toe aan je inlaad-functie!
df_input_stock = pd.read_csv(RAW_DATA_PATH / 'complete_timber_small_260310.csv', sep=';', encoding='latin1')
#df_input_stock = pd.read_csv(RAW_DATA_PATH + 'complete_timber.csv', sep=';')

# Print de kolommen om zeker te weten dat ze goed gesplitst zijn
print("Gevonden kolommen:", df_input_stock.columns.tolist())

# Controleer of alle data, inclusief de RS_0000 ID's en binaire states, goed is overgekomen
print("\nDataset succesvol ingeladen. Hier zijn de eerste elementen:\n")
print(f"Dataset bevat {df_input_stock.shape[0]} elementen\n")
print(df_input_stock.head())

Gevonden kolommen: ['Member_ID', 'State', 'Length', 'Depth', 'Width', 'E_modulus_eff', 'f_mk', 'Density', 'ECC', 'Transport_Dist', 'Emmisiefactor', 'Bewerkingsfactor']

Dataset succesvol ingeladen. Hier zijn de eerste elementen:

Dataset bevat 21 elementen

  Member_ID  State  Length  Depth  Width  E_modulus_eff  f_mk  Density    ECC  \
0  NS_00074      0  3000.0  225.0   75.0        11000.0    24      420  150.0   
1  NS_00089      0  3300.0  150.0   50.0        11000.0    24      420  150.0   
2  NS_00090      0  3300.0  150.0   75.0        11000.0    24      420  150.0   
3  NS_00091      0  3300.0  150.0  100.0        11000.0    24      420  150.0   
4  NS_00092      0  3300.0  175.0   38.0        11000.0    24      420  150.0   

   Transport_Dist  Emmisiefactor  Bewerkingsfactor  
0             500         0.0987                 0  
1             500         0.1167                 0  
2             500         0.0835                 0  
3             500         0.1436           

## Search space

De search space wordt vanuit het geometrie script geimporteerd, dan wordt een willekeurige samenstelling gekozen als beginpunt van de optimalisatie.


In [18]:
import json

json_path = 'search_space.json'
# Lees het "contract" in voor je optimizer
with open(json_path, 'r') as f:
    optimizer_search_space = json.load(f)

print(f"✅ Search Space ingeladen! De optimizer heeft {len(optimizer_search_space)} parameters om aan te draaien.")
# Nu kun je deze variabele direct aan je Optuna, PyTorch of Genetisch Algoritme voeren!

✅ Search Space ingeladen! De optimizer heeft 18 parameters om aan te draaien.


Dit script is bedoeld voor je Colab-omgeving. Het leest de search_space.json in, stelt de parameters dynamisch in op basis van jouw regels, en gebruikt een "dummy" voorspelling (waar later je echte Neurale Netwerk komt) om het beste ontwerp te vinden.

# GEOMETRY

## Random geometry

In [19]:
import json
import random
import c11_params

# ==========================================
# 1. RANDOM PARAMETERS GENEREREN (De "DNA" string)
# ==========================================
def get_random_design(json_path):
    """Leest de search space en trekt voor elke knop een willekeurige waarde."""
    with open(json_path, 'r') as f:
        search_space = json.load(f)

    random_params = {}

    for var_name, rules in search_space.items():
        if rules['type'] == 'continuous':
            # Kies een willekeurig kommagetal (bijv. voor u en v)
            random_params[var_name] = random.uniform(rules['min'], rules['max'])

        elif rules['type'] == 'discrete':
            # Kies exact één van de toegestane stapjes (bijv. voor shift_x)
            random_params[var_name] = random.choice(rules['options'])

    return random_params

# --- TEST DE FUNCTIE ---
print("Stap 1: Willekeurig DNA genereren...")
mijn_random_ontwerp = get_random_design(json_path)

print("\nSucces! Dit is het DNA van ons proef-ontwerp:")
for sleutel, waarde in mijn_random_ontwerp.items():
    print(f" - {sleutel}: \t{waarde:.3f}")

Stap 1: Willekeurig DNA genereren...

Succes! Dit is het DNA van ons proef-ontwerp:
 - v1_shift_x: 	0.750
 - v3_shift_y: 	1.125
 - v4_shift_x: 	1.125
 - v4_shift_y: 	0.375
 - v5_shift_y: 	0.750
 - v7_shift_x: 	0.375
 - v9_u: 	0.525
 - v9_v: 	0.578
 - v9_shift_z: 	-0.375
 - v10_u: 	0.264
 - v10_v: 	0.587
 - v10_shift_z: 	0.750
 - v11_u: 	0.406
 - v11_v: 	0.330
 - v11_shift_z: 	0.000
 - v12_u: 	0.274
 - v12_v: 	0.691
 - v12_shift_z: 	-1.125


# Opzet

In [20]:
import pandas as pd
import config
from geometry import generate_sample_vertices

# ==========================================
# 2. RECONSTRUCTIE FUNCTIES
# ==========================================
def reconstruct_vertices(params, sample_id=0):
    """
    Vertaalt het DNA naar exacte 3D coördinaten.
    Gebruikt de centrale motor om duplicatie van code te voorkomen.
    """
    # Waarom de logica opnieuw schrijven? We hergebruiken gewoon de motor!
    vertices_list = generate_sample_vertices(sample_id=sample_id, params=params)
    
    # We droppen de 'sample_id' kolom als je die niet nodig hebt voor een enkele reconstructie, 
    # of je laat hem staan voor consistentie.
    df = pd.DataFrame(vertices_list)
    return df

def reconstruct_edges(cells_x, cells_y):
    """
    Berekent de topologie (Edges) van het grid zonder geneste 'if' statements.
    Sneller en korter door gebruik van list comprehensions.
    """
    nodes_x_top = cells_x + 1
    nodes_y_top = cells_y + 1
    num_top = nodes_x_top * nodes_y_top
    
    edges = []

    # --- 1. TOP LAYER EDGES ---
    # Horizontale verbindingen
    edges.extend([(r * nodes_x_top + c, r * nodes_x_top + c + 1) 
                  for r in range(nodes_y_top) for c in range(cells_x)])
    # Verticale verbindingen
    edges.extend([(r * nodes_x_top + c, (r + 1) * nodes_x_top + c) 
                  for r in range(cells_y) for c in range(nodes_x_top)])

    # --- 2. BOTTOM LAYER EDGES ---
    start_bot = num_top
    # Horizontale verbindingen
    edges.extend([(start_bot + r * cells_x + c, start_bot + r * cells_x + c + 1) 
                  for r in range(cells_y) for c in range(cells_x - 1)])
    # Verticale verbindingen
    edges.extend([(start_bot + r * cells_x + c, start_bot + (r + 1) * cells_x + c) 
                  for r in range(cells_y - 1) for c in range(cells_x)])

    # --- 3. DIAGONALS (PIRAMIDE) ---
    for r in range(cells_y):
        for c in range(cells_x):
            current_bot = start_bot + r * cells_x + c
            tl, tr = r * nodes_x_top + c, r * nodes_x_top + (c + 1)
            bl, br = (r + 1) * nodes_x_top + c, (r + 1) * nodes_x_top + (c + 1)
            
            # Voeg direct alle 4 de diagonalen toe voor dit bottom-punt
            edges.extend([(current_bot, tl), (current_bot, tr), 
                          (current_bot, bl), (current_bot, br)])

    # Bouw het DataFrame in één keer op
    df_edges = pd.DataFrame(edges, columns=["V1", "V2"])
    
    # Voeg de edge_id als prefix toe
    df_edges.insert(0, "edge_id", ["e" + str(i) for i in range(len(df_edges))])
    
    return df_edges

# --- UITVOEREN VAN STAP 2 ---
print("Stap 2: DNA vertalen naar 3D Geometrie...")

# We gebruiken 'mijn_random_ontwerp' uit de VORIGE cel!
df_vertices = reconstruct_vertices(mijn_random_ontwerp)
df_edges = reconstruct_edges(c11_params.GRID_CELLS_X, c11_params.GRID_CELLS_Y)

print(f"✅ Succes! Geometrie opgebouwd met {len(df_vertices)} knooppunten en {len(df_edges)} staven.")
print("\nVoorbeeld van de gegenereerde Vertices (eerste 5):")
print(df_vertices.head())

Stap 2: DNA vertalen naar 3D Geometrie...
✅ Succes! Geometrie opgebouwd met 13 knooppunten en 32 staven.

Voorbeeld van de gegenereerde Vertices (eerste 5):
   sample_id vertex_index layer attribute      x      y    z
0          0           v0   top   support  0.000  0.000  0.0
1          0           v1   top      load  3.750  0.000  0.0
2          0           v2   top   support  6.000  0.000  0.0
3          0           v3   top      load  0.000  4.125  0.0
4          0           v4   top      load  4.125  3.375  0.0


## Feature extraction

In [21]:
import math
import pandas as pd

# ==========================================
# 3. EIGENSCHAPPEN UITREKENEN (Feature Extraction)
# ==========================================
def extract_beam_properties(df_vertices, df_edges):
    """
    Berekent staaflengtes en kent constructieve profielen toe
    op basis van de positie in het ruimtevakwerk.
    """
    # Maak een snelle zoek-dictionary voor de coördinaten
    # (Dit is veel sneller dan elke keer door de hele tabel zoeken)
    v_dict = df_vertices.set_index('vertex_index').to_dict('index')
    beams = []

    for _, edge in df_edges.iterrows():
        # Let op de 'v' prefix die we in stap 2 hebben toegevoegd
        v1_id = f"v{edge['V1']}" if not str(edge['V1']).startswith('v') else edge['V1']
        v2_id = f"v{edge['V2']}" if not str(edge['V2']).startswith('v') else edge['V2']

        pt1 = v_dict[v1_id]
        pt2 = v_dict[v2_id]

        # Stelling van Pythagoras (3D Euclidische afstand in meters)
        dx = pt1['x'] - pt2['x']
        dy = pt1['y'] - pt2['y']
        dz = pt1['z'] - pt2['z']
        lengte_m = math.sqrt(dx**2 + dy**2 + dz**2)

        # Converteer naar millimeters voor de hout-database
        lengte_mm = lengte_m * 1000.0

        # Bepaal het constructieve type en de benodigde dwarsdoorsnede (in mm)
        if pt1['layer'] == 'top' and pt2['layer'] == 'top':
            b_type = 'Top Chord'
            w_req, d_req = 75.0, 150.0   # Aangepast naar beschikbare maten (75x150)
        elif pt1['layer'] == 'bottom' and pt2['layer'] == 'bottom':
            b_type = 'Bottom Chord'
            w_req, d_req = 75.0, 100.0   # Aangepast naar 75x100
        else:
            b_type = 'Diagonal Web'
            w_req, d_req = 50.0, 100.0   # Diagonalen mogen dunner zijn (50x100)

        beams.append({
            'edge_id': edge['edge_id'],
            'type': b_type,
            'Length_Req': round(lengte_mm, 2),
            'Width_Req': w_req,
            'Depth_Req': d_req
        })

    return pd.DataFrame(beams)

# --- UITVOEREN VAN STAP 3 ---
print("Stap 3: Staaflengtes en profielen berekenen...")

# We gebruiken de tabellen uit de VORIGE cel!
df_slots = extract_beam_properties(df_vertices, df_edges)

print(f"✅ Succes! We hebben de eigenschappen van {len(df_slots)} balken berekend.")
print("\nDit is de input (Slots) voor je Cost Matrix (eerste 10 balken):")
print(df_slots.head(10))

Stap 3: Staaflengtes en profielen berekenen...
✅ Succes! We hebben de eigenschappen van 32 balken berekend.

Dit is de input (Slots) voor je Cost Matrix (eerste 10 balken):
  edge_id       type  Length_Req  Width_Req  Depth_Req
0      e0  Top Chord     3750.00       75.0      150.0
1      e1  Top Chord     2250.00       75.0      150.0
2      e2  Top Chord     4192.63       75.0      150.0
3      e3  Top Chord     1912.13       75.0      150.0
4      e4  Top Chord     3375.00       75.0      150.0
5      e5  Top Chord     2625.00       75.0      150.0
6      e6  Top Chord     4125.00       75.0      150.0
7      e7  Top Chord     3395.77       75.0      150.0
8      e8  Top Chord     3750.00       75.0      150.0
9      e9  Top Chord     1875.00       75.0      150.0


# COST MATRIX

Omdat de pseudo-LCA berekening (uit de vorige stap) de impact van de gehele balk al berekent, heb je nu een methodologische keuze:

Optie A (De dubbele boete): Je telt de LCA-score én de penalty's bij elkaar op. Hiermee straf je afval extra zwaar af ten opzichte van het puur inbrengen van materiaal. Dit dwingt het algoritme agressief naar optimaal materiaalgebruik.

Optie B (De uitsplitsing): Je berekent de LCA-score alleen over het nuttige deel, en gebruikt je nieuwe formules om het verlies in kaart te brengen.

In [22]:
# ==========================================
# CEL 1: INPUTS EN ALGEMENE PARAMETERS
# ==========================================
import pandas as pd
import numpy as np

# GWP Waarden (kg CO2 eq / kg hout)
GWP_VIRGIN = 0.50
GWP_RECLAIMED = 0.08

# LCA Parameters voor E_cost
PROCESSING_PENALTY_CO2 = 5.0  # kg CO2 boete voor bewerkingen (bijv. ontspijkeren)

print("✅ Parameters succesvol geladen.")

✅ Parameters succesvol geladen.


In [23]:
# ==========================================
# CEL 2: DE REKENFUNCTIES (MODULES)
# ==========================================

def calculate_pseudo_lca_stock(df_stock):
    """
    Berekent de pseudo-LCA score (E_cost) voor een reeds ingeladen DataFrame
    met de timber stock.
    """
    # Maak een kopie om waarschuwingen (SettingWithCopyWarning) te voorkomen
    # en de originele input dataset zuiver te houden.
    df_lca = df_stock.copy()

    print("Start in-memory pseudo-LCA berekeningen...")

    # Stap A: Bereken Volume in m3 (dimensies zijn in mm, dus delen door 1000)
    df_lca['Volume_m3'] = (df_lca['Length'] / 1000) * (df_lca['Width'] / 1000) * (df_lca['Depth'] / 1000)

    # Stap B: Material Impact (A1-A3)
    df_lca['Impact_Material_kgCO2'] = df_lca['Volume_m3'] * df_lca['ECC']

    # Stap C: Transport Impact (A4)
    # Dichtheid (Density) delen door 1000 om van kg/m3 naar ton/m3 te gaan
    df_lca['Impact_Transport_kgCO2'] = df_lca['Volume_m3'] * (df_lca['Density'] / 1000) * df_lca['Transport_Dist'] * df_lca['Emmisiefactor']

    # Stap D: Processing Impact (C3 / A3 Re-processing)
    # Binaire bewerkingsfactor (0 of 1) maal de vaste penalty
    df_lca['Impact_Processing_kgCO2'] = df_lca['Bewerkingsfactor'] * PROCESSING_PENALTY_CO2

    # Stap E: Totale E_cost Berekenen
    df_lca['E_cost_Total_kgCO2'] = (
        df_lca['Impact_Material_kgCO2'] +
        df_lca['Impact_Transport_kgCO2'] +
        df_lca['Impact_Processing_kgCO2']
    )

    return df_lca


def calculate_geometric_penalties(slot, stock_item):
    """
    Module 2: Berekent de CO2-penalty's voor zaagverlies en overdimensionering
    voor één specifieke match tussen een Slot (ontwerp) en Stock (voorraad).
    Geeft een tuple terug: (c_waste, c_overdim).
    """
    # 1. Haal afmetingen op en converteer naar meters
    l_slot = slot['Length_Req'] / 1000.0
    w_req = slot['Width_Req'] / 1000.0
    d_req = slot['Depth_Req'] / 1000.0
    a_req = w_req * d_req

    l_stock = stock_item['Length'] / 1000.0
    w_stock = stock_item['Width'] / 1000.0
    d_stock = stock_item['Depth'] / 1000.0
    a_stock = w_stock * d_stock

    rho = stock_item['Density'] # kg/m3

    # Status: 0 = Virgin, 1 = Reclaimed (afhankelijk van hoe je dataset in elkaar zit, pas dit evt. aan)
    gwp_unit = GWP_RECLAIMED if stock_item['State'] == 1 else GWP_VIRGIN

    # Zaagverlies en Overdimensionering in kg CO2 eq
    c_waste = (l_stock - l_slot) * a_stock * rho * gwp_unit
    c_overdim = (a_stock - a_req) * l_slot * rho * gwp_unit

    return max(0, c_waste), max(0, c_overdim)

print("✅ Rekenmodules succesvol gedefinieerd.")

✅ Rekenmodules succesvol gedefinieerd.


## Controle

In [24]:
# @title
# 1. Voer de functie uit op je reeds ingeladen DataFrame
df_stock_with_lca = calculate_pseudo_lca_stock(df_input_stock)

# 2. Bekijk de resultaten van de berekening
print("\nPreview van de berekende E_cost per element:")
display(df_stock_with_lca.sample(10))

Start in-memory pseudo-LCA berekeningen...

Preview van de berekende E_cost per element:


,Member_ID,State,Length,Depth,Width,E_modulus_eff,f_mk,Density,ECC,Transport_Dist,Emmisiefactor,Bewerkingsfactor,Volume_m3,Impact_Material_kgCO2,Impact_Transport_kgCO2,Impact_Processing_kgCO2,E_cost_Total_kgCO2
11,NS_00161,0,4000.0,250.0,50.0,11000.0,24,420,150.0,500,0.1225,0,0.050000,7.500000,1.286250,0.0,8.786250
12,NS_00162,0,4000.0,250.0,75.0,11000.0,24,420,150.0,500,0.1416,0,0.075000,11.250000,2.230200,0.0,13.480200
15,RS_00001,1,11746.0,588.0,168.0,9000.0,18,380,15.0,93,0.1034,1,1.160317,17.404753,4.239979,5.0,26.644732
9,NS_00159,0,4000.0,225.0,100.0,11000.0,24,420,150.0,500,0.1430,0,0.090000,13.500000,2.702700,0.0,16.202700
19,RS_00005,1,5099.0,215.0,61.0,9000.0,18,380,15.0,76,0.1002,1,0.066873,1.003101,0.193517,5.0,6.196617
5,NS_00093,0,3300.0,175.0,50.0,11000.0,24,420,150.0,500,0.1484,0,0.028875,4.331250,0.899860,0.0,5.231110
10,NS_00160,0,4000.0,250.0,38.0,11000.0,24,420,150.0,500,0.1120,0,0.038000,5.700000,0.893760,0.0,6.593760
7,NS_00147,0,4000.0,150.0,100.0,11000.0,24,420,150.0,500,0.1095,0,0.060000,9.000000,1.379700,0.0,10.379700
1,NS_00089,0,3300.0,150.0,50.0,11000.0,24,420,150.0,500,0.1167,0,0.024750,3.712500,0.606548,0.0,4.319048
2,NS_00090,0,3300.0,150.0,75.0,11000.0,24,420,150.0,500,0.0835,0,0.037125,5.568750,0.650987,0.0,6.219737


## Building of cost matrix

In [25]:
# Print even ter controle hoeveel balken erin zitten
print(f"🪵 Hout-inventaris succesvol ingeladen: {len(df_input_stock)} balken beschikbaar.")

🪵 Hout-inventaris succesvol ingeladen: 21 balken beschikbaar.


In [38]:
import numpy as np
import pandas as pd
import config

# ==========================================
# CEL 3: HOOFDSCRIPT - BOUW DE COST MATRIX
# ==========================================

def build_optimization_matrix(df_design, df_stock_raw, target_stock_ids=None):
    print("Start generatie van de integrale CO2 Cost Matrix...")

    # 1. Bereken de basis LCA score (Roep de module uit Cel 2 aan)
    df_stock = calculate_pseudo_lca_stock(df_stock_raw)

    n_slots = len(df_design)
    n_stock = len(df_stock)

    cost_matrix = np.full((n_slots, n_stock), np.inf)
    succesvolle_matches = 0
    detailed_logs = []

    # Hier slaan we de gedetailleerde berekeningen in op!
    detailed_logs = []

    for i in range(n_slots):
        slot = df_design.iloc[i]
        slot_id = slot['edge_id'] # of Element_ID, afhankelijk van je kolomnaam

        slot_max_dim = max(slot['Width_Req'], slot['Depth_Req'])
        slot_min_dim = min(slot['Width_Req'], slot['Depth_Req'])

        for j in range(n_stock):
            stock_item = df_stock.iloc[j]
            stock_id = stock_item['Member_ID']

            stock_max_dim = max(stock_item['Width'], stock_item['Depth'])
            stock_min_dim = min(stock_item['Width'], stock_item['Depth'])

            # --- HARD CONSTRAINTS (INCLUSIEF ROTATIE) ---
            fits_physically = (
                stock_item['Length'] >= slot['Length_Req'] and
                stock_max_dim >= slot_max_dim and
                stock_min_dim >= slot_min_dim
            )

            if fits_physically:
                # 2. Haal de individuele LCA componenten op
                i_mat = stock_item['Impact_Material_kgCO2']
                i_trans = stock_item['Impact_Transport_kgCO2']
                i_proc = stock_item['Impact_Processing_kgCO2']
                e_cost_base = stock_item['E_cost_Total_kgCO2']
                c_waste, c_overdim = calculate_geometric_penalties(slot, stock_item)
                total_match_score = e_cost_base + c_waste + c_overdim

                cost_matrix[i, j] = total_match_score
                succesvolle_matches += 1

                # 4. Uitgebreid loggen voor de diepte-analyse
                if target_stock_ids and stock_id in target_stock_ids:
                    detailed_logs.append({
                        'Slot_ID': slot_id,
                        'Stock_ID': stock_id,
                        'Status': '✅',
                        'Mat_CO2': round(i_mat, 3),
                        'Trans_CO2': round(i_trans, 3),
                        'Proc_CO2': round(i_proc, 3),
                        'E_cost_Base': round(e_cost_base, 2),
                        'Waste_CO2': round(c_waste, 3),
                        'Overdim_CO2': round(c_overdim, 3),
                        'TOTAL_Score': round(total_match_score, 2)
                    })
            else:
                if target_stock_ids and stock_id in target_stock_ids:
                    detailed_logs.append({
                        'Slot_ID': slot_id, 'Stock_ID': stock_id, 'Status': '❌',
                        'Mat_CO2': '-', 'Trans_CO2': '-', 'Proc_CO2': '-', 'E_cost_Base': "-",
                        'Waste_CO2': '-', 'Overdim_CO2': '-', 'TOTAL_Score': np.inf
                    })

    print(f"✅ Matrix gegenereerd! Dimensies: {n_slots} benodigde staven x {n_stock} inventaris-balken.")
    print(f"📊 Aantal fysiek geldige combinaties gevonden: {succesvolle_matches}")

    return cost_matrix, df_stock, pd.DataFrame(detailed_logs)

# --- UITVOEREN VAN DE WORKFLOW MET ECHTE DATA ---
# Bepaal automatisch twee balken om te onderzoeken (1x RS en 1x NS)
rs_balken = df_input_stock[df_input_stock['Member_ID'].str.contains('RS')]['Member_ID'].tolist()
ns_balken = df_input_stock[df_input_stock['Member_ID'].str.contains('NS')]['Member_ID'].tolist()

balk_onderzoek = []
if rs_balken: balk_onderzoek.append(rs_balken[0]) # Pak de eerste Reclaimed balk
if ns_balken: balk_onderzoek.append(ns_balken[0]) # Pak de eerste New balk

print(f"🔍 Diepte-analyse gestart voor: {balk_onderzoek}")

# Bouw de matrix en haal het logboek (df_logs) op
cost_matrix, verrijkte_stock, df_logs = build_optimization_matrix(df_slots, df_input_stock, target_stock_ids=balk_onderzoek)

# 3. Presentatie van het resultaat
# Maak er een DataFrame van, puur om het netjes te kunnen printen.
df_cost_matrix_display = pd.DataFrame(
    cost_matrix,
    index=[f"{row['edge_id']} ({row['Length_Req']:.0f}mm)" for _, row in df_slots.iterrows()],
    columns=verrijkte_stock['Member_ID'].tolist()
)

df_cost_matrix_display.to_csv(config.EXPORT_PATH /'final_cost_matrix.csv', index=True)

# --- PRESENTEER DE DIEPTE-ANALYSE ---
print("\n" + "="*80)
print("🔬 DIEPTE-ANALYSE: ONLEDING VAN DE CO2 PENALTY")
print("="*80)

# Sorteer het logboek netjes op Stock_ID
if not df_logs.empty:
    df_logs = df_logs.sort_values(by=['Stock_ID', 'Slot_ID']).reset_index(drop=True)
    # Print als nette tabel, verwijder de NaN voor overzichtelijkheid
    print(df_logs.fillna('-').to_string(index=False))
else:
    print("Geen logboek data gevonden.")

print("="*80)

print("\nPreview Cost Matrix (eerste 8 staven, eerste 5 inventaris-balken):")
print("(Let op: 'inf' betekent dat de inventaris-balk te klein is en wordt uitgesloten)\n")
print("=" *80)
print(df_cost_matrix_display.iloc[:8, :5].round(2))

🔍 Diepte-analyse gestart voor: ['RS_00001', 'NS_00074']
Start generatie van de integrale CO2 Cost Matrix...
Start in-memory pseudo-LCA berekeningen...
✅ Matrix gegenereerd! Dimensies: 32 benodigde staven x 21 inventaris-balken.
📊 Aantal fysiek geldige combinaties gevonden: 368

🔬 DIEPTE-ANALYSE: ONLEDING VAN DE CO2 PENALTY
Slot_ID Stock_ID Status Mat_CO2 Trans_CO2 Proc_CO2 E_cost_Base Waste_CO2 Overdim_CO2  TOTAL_Score
     e0 NS_00074      ❌       -         -        -           -         -           -          inf
     e1 NS_00074      ✅   7.594     1.049      0.0        8.64     2.658       2.658        13.96
    e10 NS_00074      ✅   7.594     1.049      0.0        8.64     0.957       3.225        12.82
    e11 NS_00074      ✅   7.594     1.049      0.0        8.64     2.658       2.658        13.96
    e12 NS_00074      ✅   7.594     1.049      0.0        8.64     1.155       5.264        15.06
    e13 NS_00074      ✅   7.594     1.049      0.0        8.64     0.019       5.896   

# MATCHING ALGORITHM / ILP

## Script

In [39]:
import pulp
import numpy as np
import pandas as pd

# ==========================================
# CEL 4: TOEPASSING VAN HET OPTIMALISATIE ALGORITME (MILP)
# ==========================================
print("Start MILP Optimizer voor definitieve toewijzing...")

# 1. DATA KOPPELEN VANUIT VORIGE SCRIPTS
# Haal de namen op uit de DataFrames van de vorige cellen
stock_items = verrijkte_stock['Member_ID'].tolist()
construction_slots = df_slots['edge_id'].tolist()

# Dynamische categorisatie (kijkt naar 'NS' en 'RS' in de Member_ID)
new_items = [item for item in stock_items if 'NS' in item]
reclaimed_items = [item for item in stock_items if 'RS' in item]

print(f"📊 Inventaris: {len(reclaimed_items)} Reclaimed, {len(new_items)} New elementen.")
print(f"📐 Constructie vereist: {len(construction_slots)} staven.")

# 2. FILTEREN VAN DE COST MATRIX (Sparse Matrix Maken)
# We maken alleen combinaties aan die fysiek passen (waar de cost NIET oneindig is)
valid_matches = []
costs = {}

for i, slot_id in enumerate(construction_slots):
    for j, stock_id in enumerate(stock_items):
        cost = cost_matrix[i, j]
        # Als de cost niet 'inf' is, is het een geldige match!
        if cost != np.inf:
            valid_matches.append((stock_id, slot_id))
            costs[(stock_id, slot_id)] = cost

print(f"⚡ Aantal geldige combinaties voor de solver gereduceerd tot: {len(valid_matches)}")

# 3. HET MODEL OPZETTEN
prob = pulp.LpProblem("Reclaimed_Timber_Matching", pulp.LpMinimize)

# 4. DECISION VARIABLES
# We maken uitsluitend 'knoppen' aan voor de combinaties die kunnen passen
x = pulp.LpVariable.dicts("Match", valid_matches, 0, 1, pulp.LpBinary)

# 5. OBJECTIVE FUNCTION (Doel: Minimaliseer de totale CO2 penalty)
prob += pulp.lpSum([x[match] * costs[match] for match in valid_matches])

# 6. CONSTRAINTS (De Regels)

# Regel A: Elke staaf in de constructie MOET precies 1 balk toegewezen krijgen
for slot_id in construction_slots:
    # Zoek alle balken die überhaupt passen op dit slot
    valid_stock_for_slot = [stock_id for (stock_id, s_id) in valid_matches if s_id == slot_id]

    if not valid_stock_for_slot:
        print(f"⚠️ FATALE FOUT: Slot {slot_id} heeft GEEN ENKELE fysiek passende balk in de hele voorraad!")
    else:
        prob += pulp.lpSum([x[(stock_id, slot_id)] for stock_id in valid_stock_for_slot]) == 1

# Regel B: Reclaimed hout is uniek (Max 1x gebruiken)
for stock_id in reclaimed_items:
    valid_slots_for_stock = [s_id for (s_id_tuple, s_id) in valid_matches if s_id_tuple == stock_id]
    if valid_slots_for_stock:
        prob += pulp.lpSum([x[(stock_id, slot_id)] for slot_id in valid_slots_for_stock]) <= 1

# Regel C: Nieuw hout mag oneindig gebruikt worden (Limiet = aantal benodigde staven)
for stock_id in new_items:
    valid_slots_for_stock = [s_id for (s_id_tuple, s_id) in valid_matches if s_id_tuple == stock_id]
    if valid_slots_for_stock:
        prob += pulp.lpSum([x[(stock_id, slot_id)] for slot_id in valid_slots_for_stock]) <= len(construction_slots)

# ==========================================
# 7. OPLOSSEN EN RESULTATEN
# ==========================================
prob.solve()

print("\n" + "="*50)
print(f"STATUS OPLOSSING: {pulp.LpStatus[prob.status]}")
print("="*50)

if pulp.LpStatus[prob.status] == 'Optimal':
    total_cost = pulp.value(prob.objective)

    # Sla de winnende combinaties op in een overzichtelijke lijst
    results = []
    for j in construction_slots:
        for i in stock_items:
            match = (i, j)
            if match in x and x[match].varValue == 1:
                results.append({'Slot_ID': j, 'Assigned_Stock': i, 'CO2_Penalty': round(costs[match], 2)})

    df_results = pd.DataFrame(results)

    print(f"\n✅ Het optimale ontwerp is gevonden met een totale CO2 penalty van {total_cost:.2f} kg!")

    # ==========================================
    # 8. MERGEN MET DE ORIGINELE EDGE MATRIX
    # ==========================================
    # We gebruiken pd.merge om de nieuwe 'assigned_timber' kolom naast je bestaande V1/V2 kolommen te plakken
    # Let op: Vervang 'df_slots' door de naam van je originele edge dataframe als die anders heet (bijv. df_opt_edges)
    
    df_final_edges = pd.merge(df_slots, df_results[['edge_id', 'assigned_timber']], on='edge_id', how='left')
    
    print("\nDefinitieve Hout-Toewijzing:")
    print("-" * 50)
    print(df_results.to_string(index=False))

    # Optioneel: Exporteer dit naar CSV om in Grasshopper te kleuren
    # df_results.to_csv('definitieve_toewijzing.csv', index=False)
    
else:
    print("❌ Het algoritme kon geen oplossing vinden. Waarschijnlijk is je voorraad niet toereikend voor de opgevraagde constructie.")

Start MILP Optimizer voor definitieve toewijzing...
📊 Inventaris: 6 Reclaimed, 15 New elementen.
📐 Constructie vereist: 32 staven.
⚡ Aantal geldige combinaties voor de solver gereduceerd tot: 368

STATUS OPLOSSING: Optimal

✅ Het optimale ontwerp is gevonden met een totale CO2 penalty van 390.04 kg!


KeyError: "None of [Index(['edge_id', 'assigned_timber'], dtype='str')] are in the [columns]"

## 1D Cutting stock problem

In [28]:
# @title
import pulp
import numpy as np
import pandas as pd

print("Start Advanced MILP Optimizer (Inclusief 1D Cutting Stock / Nesting)...")

# ==========================================
# 1. DATA VOORBEREIDEN (Vanuit df_slots en verrijkte_stock)
# ==========================================
# Maak dictionaries zodat de Optimizer razendsnel lengtes en dimensies kan opzoeken
slot_lengths = {row['edge_id']: row['Length_Req'] for _, row in df_slots.iterrows()}
slot_max_dim = {row['edge_id']: max(row['Width_Req'], row['Depth_Req']) for _, row in df_slots.iterrows()}
slot_min_dim = {row['edge_id']: min(row['Width_Req'], row['Depth_Req']) for _, row in df_slots.iterrows()}
slot_area = {row['edge_id']: (row['Width_Req'] * row['Depth_Req']) / 1e6 for _, row in df_slots.iterrows()} # Area in m2

stock_lengths = {row['Member_ID']: row['Length'] for _, row in verrijkte_stock.iterrows()}
stock_max_dim = {row['Member_ID']: max(row['Width'], row['Depth']) for _, row in verrijkte_stock.iterrows()}
stock_min_dim = {row['Member_ID']: min(row['Width'], row['Depth']) for _, row in verrijkte_stock.iterrows()}
stock_area = {row['Member_ID']: (row['Width'] * row['Depth']) / 1e6 for _, row in verrijkte_stock.iterrows()} # Area in m2
stock_rho = {row['Member_ID']: row['Density'] for _, row in verrijkte_stock.iterrows()}

# Bepaal GWP voor elk hout-item (Afhankelijk of NS of RS in de naam staat)
stock_gwp = {row['Member_ID']: GWP_RECLAIMED if 'RS' in row['Member_ID'] else GWP_VIRGIN for _, row in verrijkte_stock.iterrows()}
stock_base_cost = {row['Member_ID']: row['E_cost_Total_kgCO2'] for _, row in verrijkte_stock.iterrows()}

stock_items = verrijkte_stock['Member_ID'].tolist()
construction_slots = df_slots['edge_id'].tolist()
new_items = [item for item in stock_items if 'NS' in item]
reclaimed_items = [item for item in stock_items if 'RS' in item]

# ==========================================
# 2. FILTEREN VAN GELDIGE COMBINATIES (Alleen Dwarsdoorsnede & Max Lengte)
# ==========================================
valid_matches = []
for slot_id in construction_slots:
    for stock_id in stock_items:
        # HARD CONSTRAINT: Balk moet in ieder geval dik/breed genoeg zijn (inclusief rotatie)
        # EN de stock-balk moet langer of gelijk zijn aan deze ene specifieke staaf.
        if (stock_max_dim[stock_id] >= slot_max_dim[slot_id] and
            stock_min_dim[stock_id] >= slot_min_dim[slot_id] and
            stock_lengths[stock_id] >= slot_lengths[slot_id]):
            valid_matches.append((stock_id, slot_id))

# ==========================================
# 3. HET MILP MODEL OPZETTEN
# ==========================================
prob = pulp.LpProblem("Timber_Cutting_Stock_Optimization", pulp.LpMinimize)

# Variabele 1: Wordt deze combinatie (Slot uit Stock) gemaakt? (0 of 1)
x = pulp.LpVariable.dicts("Match", valid_matches, 0, 1, pulp.LpBinary)

# Variabele 2: Wordt deze Hout-balk überhaupt AANGESNEDEN? (0 of 1)
# (Dit is nieuw! Hiermee bepalen we of we het basisafval en base_cost moeten rekenen)
y = pulp.LpVariable.dicts("UsedStock", stock_items, 0, 1, pulp.LpBinary)

# ==========================================
# 4. OBJECTIVE FUNCTION (CO2 Minimalisatie Dynamisch Berekenen)
# ==========================================
objective_terms = []

# A. De Base LCA Cost (alleen als we de balk aansnijden)
for stock_id in stock_items:
    objective_terms.append(y[stock_id] * stock_base_cost[stock_id])

# B. De Overdimensionering (Wordt berekend per stukje staaf dat we uit de balk zagen)
for stock_id, slot_id in valid_matches:
    lengte_m = slot_lengths[slot_id] / 1000.0
    area_verschil = stock_area[stock_id] - slot_area[slot_id]
    overdim_co2 = area_verschil * lengte_m * stock_rho[stock_id] * stock_gwp[stock_id]

    objective_terms.append(x[(stock_id, slot_id)] * max(0, overdim_co2))

# C. Het Zaagverlies (Restlengte). Dit is de genialiteit van 1D-CSP:
# (Totale lengte van de balk - Som van alle stukjes die we eruit zagen) * Area * Density * GWP
for stock_id in stock_items:
    valid_slots_for_this_stock = [s_id for (st_id, s_id) in valid_matches if st_id == stock_id]

    if valid_slots_for_this_stock:
        # Lengte in meters!
        totale_lengte_m = stock_lengths[stock_id] / 1000.0
        gebruikte_lengte_m = pulp.lpSum([x[(stock_id, slot_id)] * (slot_lengths[slot_id] / 1000.0) for slot_id in valid_slots_for_this_stock])

        # De CO2 waarde van een meter 'leeg' hout van deze balk
        co2_per_meter_waste = stock_area[stock_id] * stock_rho[stock_id] * stock_gwp[stock_id]

        # Voeg de afval berekening toe: (Lengte Balk * IsGebruikt - Gebruikte Lengte) * CO2_per_m
        waste_term = (totale_lengte_m * y[stock_id] - gebruikte_lengte_m) * co2_per_meter_waste
        objective_terms.append(waste_term)

# Voeg alles samen als het doel van de AI
prob += pulp.lpSum(objective_terms)

# ==========================================
# 5. CONSTRAINTS (De Regels)
# ==========================================
# Regel 1: Elke staaf in de constructie MOET precies 1 keer gezaagd worden
for slot_id in construction_slots:
    prob += pulp.lpSum([x[(st_id, slot_id)] for (st_id, sl_id) in valid_matches if sl_id == slot_id]) == 1

# Regel 2: De stukken die we uit een balk zagen, mogen samen NIET langer zijn dan de balk! (De 1D Bin Packing regel)
for stock_id in stock_items:
    valid_slots_for_this_stock = [s_id for (st_id, s_id) in valid_matches if st_id == stock_id]
    if valid_slots_for_this_stock:
        # Let op: We vermenigvuldigen de maximale lengte met y[stock_id] (0 of 1).
        # Als y 0 is (niet gebruikt), mag de som dus ook maximaal 0 zijn!
        prob += pulp.lpSum([x[(stock_id, slot_id)] * slot_lengths[slot_id] for slot_id in valid_slots_for_this_stock]) <= stock_lengths[stock_id] * y[stock_id]

# Regel 3: Reclaimed hout kan maar 1 keer fysiek "aangesneden" worden (y <= 1).
# Voor New hout geldt dat we er theoretisch oneindig veel van kunnen "kopen" (als ze exact deze lengte hebben),
# MAAR omdat we hier 'nesting' doen binnen specifieke ID's uit je lijst, is y een binaire variabele voor ELKE balk.
# De limieten worden dus impliciet beheerd doordat elke balk_id maar 1 object in je voorraad is.

# ==========================================
# 6. OPLOSSEN
# ==========================================
prob.solve()

print("\n" + "="*50)
print(f"STATUS OPLOSSING: {pulp.LpStatus[prob.status]}")
print("="*50)

if pulp.LpStatus[prob.status] == 'Optimal':
    total_cost = pulp.value(prob.objective)

    print(f"\n✅ Optimaal 'Nesting' ontwerp gevonden! CO2 Penalty: {total_cost:.2f} kg")

    print("\n📦 HOE DE BALKEN WORDEN OPGEZAAGD:")
    print("-" * 50)
    for stock_id in stock_items:
        if y[stock_id].varValue == 1:
            # Welke staven zijn uit deze balk gehaald?
            gekozen_slots = [s_id for (st_id, s_id) in valid_matches if st_id == stock_id and x[(stock_id, s_id)].varValue == 1]
            if gekozen_slots:
                totale_gebruikte_lengte = sum([slot_lengths[s_id] for s_id in gekozen_slots])
                restafval = stock_lengths[stock_id] - totale_gebruikte_lengte

                print(f"Balk: {stock_id} (Lengte: {stock_lengths[stock_id]:.0f}mm) -> Aangesneden!")
                print(f"  └─ Bevat staven: {', '.join(gekozen_slots)}")
                print(f"  └─ Restafval: {restafval:.0f}mm")
else:
    print("❌ Geen oplossing. De voorraad (zélfs met opdelen) is niet voldoende.")

Start Advanced MILP Optimizer (Inclusief 1D Cutting Stock / Nesting)...

STATUS OPLOSSING: Infeasible
❌ Geen oplossing. De voorraad (zélfs met opdelen) is niet voldoende.


# EXPORT

This is where the best parameters of the sctructure are exported, this is done in a format that can be used by grasshopper script to translate it into geometry

In [37]:
# @title
import pandas as pd
import config
import c11_params  # <-- TOEGEVOEGD: Vergeet deze import niet!
from geometry import generate_sample_vertices, generate_edges

# ==========================================
# 5. GEOMETRISCHE RECONSTRUCTIE VAN HET OPTIMUM
# ==========================================
print("\n📦 Exporteren van het volledige winnende ontwerp naar CSV's...")

# Haal de winnende getallen op (de 'DNA' vector)
# <-- AANGEPAST: Moet een dictionary zijn, geen getal '1'
best_parameters = {} # Vervang {} later door: optimizer.best_params
OPTIMUM_ID = 9999

def reconstruct_optimum_vertices(params, sample_id):
    """Herbouwt exact dat ene beste ontwerp."""
    # <-- AANGEPAST: params nu correct doorgegeven aan de motor
    vertices = generate_sample_vertices(sample_id, params=params)
    return pd.DataFrame(vertices)

# --- 2. Genereer de Vertices ---
df_opt_vertices = reconstruct_optimum_vertices(best_parameters, OPTIMUM_ID)

# --- 3. Genereer de Edges ---
# <-- AANGEPAST: num_samples op 1 gezet, want we bouwen maar 1 optimum!
df_opt_edges = generate_edges(num_samples=1, cells_x=c11_params.GRID_CELLS_X, cells_y=c11_params.GRID_CELLS_Y)

# BELANGRIJK: Omdat generate_edges() altijd bij sample_id=0 begint,
# forceren we de ID naar ons OPTIMUM_ID, zodat hij synchroon loopt met de vertices.
df_opt_edges['sample_id'] = OPTIMUM_ID

# --- 4. Exporteren ---
pad_vertices = config.EXPORT_PATH / 'optimum_vertices.csv'
pad_edges = config.EXPORT_PATH / 'optimum_edges.csv'

df_opt_vertices.to_csv(pad_vertices, index=False)
df_opt_edges.to_csv(pad_edges, index=False)

print(f"✅ Succes! Volledig winnende geometrie ({len(df_opt_vertices)} punten, {len(df_opt_edges)} lijnen) opgeslagen.")
print(f"   -> {pad_vertices}")
print(f"   -> {pad_edges}")


📦 Exporteren van het volledige winnende ontwerp naar CSV's...
✅ Succes! Volledig winnende geometrie (13 punten, 32 lijnen) opgeslagen.
   -> C:\Users\jaspe\OneDrive\06 Building Technology TU\2.2 - 2.4\60_Research_Exports\optimum_vertices.csv
   -> C:\Users\jaspe\OneDrive\06 Building Technology TU\2.2 - 2.4\60_Research_Exports\optimum_edges.csv
